# DalMax — Google Colab Pro runbook (notebook form)

Runnable companion to [`COLAB_RUNBOOK.md`](https://github.com/MarioCarvalhoBr/dalmax-deep-active-learning-python/blob/main/COLAB_RUNBOOK.md) — every cell below
mirrors a numbered section of that runbook. **`COLAB_RUNBOOK.md` is the
source of truth**; if this notebook and the runbook ever disagree, trust the
runbook (and re-sync this notebook).

**Never run this notebook on the local no-GPU dev notebook**
(`.claude/rules/data-safety.md`) — it is for Google Colab Pro only.

Run every cell top to bottom, in order. The **Parameters** cell right below
is the only one you should normally need to edit.

In [ ]:
import os

# Parameters -- the ONLY cell in this notebook you should need to edit.
DRIVE_ROOT = "/content/drive/MyDrive/UFMS/Pós-graduação/Doutorado/FINAL/PROJETO/DALMAX"
REPO_URL = "https://github.com/MarioCarvalhoBr/dalmax-deep-active-learning-python.git"
REPO_DIR = "/content/dalmax"
BRANCH = "main"
METHOD = "rnhal"  # or "texhal" -- which ablation suite section 6/7 run (see
                   # files_config/ablations/README.md: rnhal = paper 3 (SSRAE,
                   # already executed), texhal = paper 2 (VCTex, not yet run)

PART = "all"  # which campaign part section 5b runs: all | paper1 | upper_bound | rnhal | texhal
              # (comma-separated allowed, e.g. "rnhal,texhal") -- see COLAB_RUNBOOK.md "A100 full campaign"

# Also export as env vars so every `!`-shell cell below (and
# scripts/colab/setup_colab.sh's own DRIVE_ROOT override) sees them too.
os.environ["DRIVE_ROOT"] = DRIVE_ROOT
os.environ["METHOD"] = METHOD
os.environ["PART"] = PART

print("DRIVE_ROOT:", DRIVE_ROOT)
print("REPO_URL:  ", REPO_URL)
print("REPO_DIR:  ", REPO_DIR)
print("BRANCH:    ", BRANCH)
print("METHOD:    ", METHOD)
print("PART:      ", PART)

## 0. Runtime check

See `COLAB_RUNBOOK.md` section 0. Expect one GPU listed (T4/L4/A100
depending on what Colab Pro assigns this session).

In [ ]:
!nvidia-smi

In [ ]:
import sys

_supported = {(3, 10), (3, 11), (3, 12)}
if sys.version_info[:2] not in _supported:
    raise RuntimeError(
        f"Python {sys.version_info.major}.{sys.version_info.minor} detected, "
        "but torch==2.5.0 (pinned in poetry.lock) has no wheels for Python "
        "3.13+. Stop here -- see COLAB_RUNBOOK.md section 8's Python-version "
        "troubleshooting entry before proceeding; do not force-install a "
        "different Python inside the Colab image."
    )
print(f"Python {sys.version_info.major}.{sys.version_info.minor} OK")

## 1. Mount Google Drive

See `COLAB_RUNBOOK.md` section 1.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
assert os.path.isdir(f"{DRIVE_ROOT}/DATA/daninhas_full"), (
    f"{DRIVE_ROOT}/DATA/daninhas_full not found. Check DRIVE_ROOT in the "
    "Parameters cell above, and that daninhas_full was uploaded to Drive at "
    "that path -- every later step assumes it (COLAB_RUNBOOK.md section 1)."
)
print("Dataset folder found on Drive.")

In [ ]:
!ls "$DRIVE_ROOT/DATA/daninhas_full" | head

## 2. Clone or pull the repo

Idempotent: clones fresh into `/content/dalmax` (local runtime disk, not
Drive -- see `COLAB_RUNBOOK.md`'s hybrid-layout decision) the first time, or
pulls `BRANCH` if this runtime already has it (rare -- Colab runtimes are
usually fresh each session, but harmless to check). See
`COLAB_RUNBOOK.md` section 2.

In [ ]:
import subprocess

if os.path.isdir(os.path.join(REPO_DIR, ".git")):
    print(f"{REPO_DIR} already exists -- pulling latest {BRANCH}...")
    subprocess.run(["git", "-C", REPO_DIR, "pull", "origin", BRANCH], check=True)
else:
    print(f"Cloning {REPO_URL} (branch {BRANCH}) into {REPO_DIR}...")
    subprocess.run(
        ["git", "clone", "--branch", BRANCH, REPO_URL, REPO_DIR], check=True
    )

# os.chdir affects `!`-cells too (they share the process cwd), but `%cd` is
# the documented/safe way to also update Colab's own displayed cwd state.
os.chdir(REPO_DIR)
%cd $REPO_DIR
!git log --oneline -1

## 3. Install dependencies with Poetry

See `COLAB_RUNBOOK.md` section 3.

In [ ]:
!pip install -q poetry
!poetry install

This is a fresh runtime each session, so `poetry install` re-downloads the
pinned `torch==2.5.0`/`torchvision==0.20.0` CUDA wheels every time -- expect
roughly **5 minutes** (not previously measured on Colab specifically; treat
as an estimate). It creates an in-project `.venv/` on the local runtime disk,
same as the lab machine and local notebook.

**Alternative (not recommended by default)**:
`poetry config virtualenvs.create false --local` installs into the runtime's
system Python instead -- slightly faster, but downgrades Colab's preinstalled
torch build to the pinned 2.5.0 and loses environment isolation. See
`COLAB_RUNBOOK.md` section 3 for the full trade-off.

In [ ]:
!poetry run python -c "import torch; print(torch.cuda.is_available(), torch.cuda.device_count(), torch.cuda.get_device_name(0))"

## 4. Wire up the dataset and results/ (Drive)

`make colab-setup` runs `scripts/colab/setup_colab.sh`: it verifies Drive is
mounted, builds `daninhas_full.zip` on Drive on the **first session only**
(later sessions reuse it), unzips it onto the local runtime disk for fast
reads, and symlinks `results/` to Drive so every artifact survives a
disconnect. Idempotent -- safe to re-run after a disconnect. See
`COLAB_RUNBOOK.md` section 4.

In [ ]:
!make colab-setup

## 5. Sanity checks

See `COLAB_RUNBOOK.md` section 5.

In [ ]:
!make test

In [ ]:
!make smoke

In [ ]:
!make colab-check

In [ ]:
!poetry run python loader.py --model results/colab_check/daninhas_full/SEED_1/NQ_100_NIL_100_NR_1_NE_10/SSRAEKmeansHCSampling/saved_model.pth

In [ ]:
import glob

img = sorted(glob.glob("DATA/daninhas_full/test/DATASET_GRAMINEA/*"))[0]
print(img)
!poetry run python predict.py --model results/colab_check/daninhas_full/SEED_1/NQ_100_NIL_100_NR_1_NE_10/SSRAEKmeansHCSampling/saved_model.pth --image "{img}"

**Only continue to section 6 once all of the above pass.**

## 5b. A100 full campaign (2026-09-29) -- the recommended path for the final re-execution

One manifest drives the single, deduplicated re-execution of everything the three papers need
(paper 1: 12 classical strategies + KMH x nq {10,50,100} + the `FullSupervised` upper bound;
paper 2/3: TexHAL/RNHAL ablations at nq=100): **192 runs / 64 run groups**. Select an **A100**
runtime first (section 0). See `COLAB_RUNBOOK.md` "A100 full campaign" and
`.specs/experiments/campaign-a100.md`. Sections 6/7 below (ablation-only) are kept for partial runs.
The `PART` parameter selects `all` or a subset (split across sessions by launching one part per session).

In [ ]:
!make campaign-list PART={PART}

Run (skip-existing: **re-running this cell resumes after a disconnect** -- redo sections 0-4 first).
The first job is the cheapest one; once it finishes, run the environment check cell below **before
leaving the batch unattended** (`environment.gpus[0].name` must say A100).

In [ ]:
import time

start = time.time()
!make campaign-run PART={PART}

In [ ]:
import glob
import json

hits = glob.glob("results/campaign_a100/shared/random_nq100/*/SEED_1/*/RandomSampling/run_metadata.json")
if hits:
    meta = json.load(open(hits[0]))
    print("GPU:", meta["environment"]["gpus"][0]["name"])
    print("determinism:", meta["determinism"])
else:
    print("first job (shared/random_nq100 seed 1) has not finished yet")

In [ ]:
elapsed_hours = (time.time() - start) / 3600
print(f"Campaign part(s) {PART} took {elapsed_hours:.2f} hours on this runtime.")
print("Paste this number into COLAB_RUNBOOK.md's A100 campaign section (wall-clock TBD).")

**Monitoring** (from a separate cell while the campaign is running, or after a disconnect):

In [ ]:
!tail -n 30 results/campaign_a100/campaign.log

In [ ]:
!cat results/campaign_a100/failures.log

**Verify**: OK/INCOMPLETE/MISSING per job plus the seed-consistency audit (every run of a seed must start
from the identical initial labeled set). Re-run the run cell above to redo failed/missing jobs.

In [ ]:
!make campaign-verify PART={PART}

**Report** + copy to Drive (tables md/tex/csv, `seed_audit.md`, mean confusion matrices).

In [ ]:
!make campaign-report

In [ ]:
!mkdir -p "$DRIVE_ROOT/results"
!cp -r docs/results/campaign_a100 "$DRIVE_ROOT/results/campaign_a100_tables"

In [ ]:
!ls "$DRIVE_ROOT/results/campaign_a100_tables"

## 6. Run the ablation study (ablation-only path; the campaign in 5b is recommended for the final re-execution)

Runs the suite selected by the Parameters cell's `METHOD` (`rnhal` = paper 3,
SSRAE, already executed; `texhal` = paper 2, VCTex, not yet run -- see
`files_config/ablations/README.md`). Same batch as `LAB_RUNBOOK.md` section 2
(11 configs / 33 runs for rnhal, 12 configs / 36 runs for texhal), but
sequentially on Colab's **one** GPU instead of split across two lab GPUs --
expect roughly **2x** the per-GPU lab wall-clock time as a rough estimate for
a suite with no measured Colab number yet (rnhal: measured, ~5h15m, see
`COLAB_RUNBOOK.md` section 6; texhal: **TBD**, the next cell times this run
so you can record it). See `COLAB_RUNBOOK.md` section 6 for the full
disconnect policy and Drive quota notes.

In [ ]:
import time

start = time.time()
!make ablations-colab METHOD={METHOD}

In [ ]:
elapsed_hours = (time.time() - start) / 3600
print(f"Ablation sweep (METHOD={METHOD}) took {elapsed_hours:.2f} hours on this Colab GPU.")
print(
    "Paste this number into COLAB_RUNBOOK.md section 6 and LAB_RUNBOOK.md's "
    "TBD wall-clock note (rnhal is already measured; texhal is the one "
    "still TBD as of this writing)."
)

**Monitoring** (run from a separate cell while the sweep above is still running, or after a disconnect):

In [ ]:
!tail -n 30 results/ablations/{METHOD}/colab.log

In [ ]:
!cat results/ablations/{METHOD}/gpu0_failures.log results/ablations/{METHOD}/gpu1_failures.log

**Resume after disconnect**: re-run the Parameters cell (same `METHOD`) and sections 0-4
above (all idempotent), then re-run the `!make ablations-colab METHOD={METHOD}` cell --
`SKIP_EXISTING=1` (default) skips every `(study, config, seed)` triple that
already has a `results.json` on Drive and only executes the remaining ones.

In [ ]:
# Running a single triple manually (e.g. to retry one that failed -- see
# monitoring cells above). Edit STUDY/CONFIG/SEED, then run this cell.
# Uses the Parameters cell's METHOD.
STUDY = "6_2"
CONFIG = "hier_L2a"
SEED = 2

!CUDA_VISIBLE_DEVICES=0 poetry run python trainer.py \
    --params_json files_config/ablations/{METHOD}/{CONFIG}.json \
    --dataset_name=DANINHAS \
    --strategy_name RepresentationStrategy \
    --n_query 100 \
    --seed {SEED} \
    --n_round 8 \
    --dir_results=results/ablations/{METHOD}/{STUDY}/{CONFIG}/ \
    --device cuda

## 7. Collect results

See `COLAB_RUNBOOK.md` section 7.

In [ ]:
!make ablation-report METHOD={METHOD}

In [ ]:
!mkdir -p "$DRIVE_ROOT/results/ablation_tables"
!cp -r docs/results/ablation_tables/{METHOD} "$DRIVE_ROOT/results/ablation_tables/{METHOD}"

In [ ]:
!ls "$DRIVE_ROOT/results/ablation_tables/{METHOD}"

**Committing from Colab requires git credentials** that this runbook does not
set up by default. Recommended: the `cp` above already copied the tables to
Drive -- pull them down on your local notebook and commit/push from there
per `.claude/rules/git-workflow.md` (push requires your explicit
confirmation). Alternative (configuring a GitHub token inside this Colab
session to commit/push directly): see `COLAB_RUNBOOK.md` section 7.

## 8. Troubleshooting

See `COLAB_RUNBOOK.md` section 8 for full detail. Quick pointers:

- Drive FUSE slowness / `Transport endpoint is not connected` -> remount
  Drive (section 1) and re-run `!make colab-setup`.
- `results/` symlink or local `results/` conflicts -> read
  `scripts/colab/setup_colab.sh`'s error message; it names the exact
  conflicting path.
- OOM on a T4 -> copy the params JSON into gitignored `files_config/local/`
  and lower `batch_size` there; never edit `files_config/ablations/` in
  place.
- Session limit reached -> this is exactly what section 6's
  `SKIP_EXISTING`-based resume flow is for.
- Python 3.13+ runtime -> not a supported path; see section 8's entry before
  doing anything else.